# 4. M2: InLegalBERT similarity to lawyer highlighted exemplars

**Reads:** `intermediate/sentences.pkl`, `intermediate/gold_sentences.csv`.
**Writes:** the embedding cache in `intermediate/`, and M2 predictions in `predictions/`: on the gold sentences (dev and test), and on every sentence for `07_corpus_outcomes`.

Exemplars and thresholds come only from the dev judgments. Test judgments are scored once, with the final model.

The first run downloads InLegalBERT and encodes every sentence, which takes a while. Later runs load the cache.

## Setup

In [1]:
from argmine_common import *

PROTOTYPE_MODE = "max"           # "max" = nearest exemplar, "mean" = category centroid
USE_DEFINITIONS = True           # add the coded category definitions as exemplars
EXCLUDE_QUOTED_EXEMPLARS = True  # skip quoted statute or precedent text as exemplars

sents = load_sentences()
GOLD_IDX, Y, DOCS, gold_docs, TEST = load_gold()

## Sentence embeddings (InLegalBERT)

InLegalBERT is pretrained on Indian court judgments (Paul et al., 2023). We mean pool the last layer, then center the vectors on the corpus mean before cosine similarity. Raw BERT vectors all point in a similar direction, so uncentered cosine scores bunch near 1 and separate poorly. The encoder and the embedding cache are in `argmine_common.py`; embeddings are cached in `argument_mining/intermediate/`.

In [2]:
X, post = load_embeddings(sents)
DEF_X = post(encode([CATEGORY_DEFINITIONS[c] for c in CATS])) if USE_DEFINITIONS else None
print("Embeddings:", X.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 39050.55it/s]


[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings: (9686, 768)


## M2: similarity to lawyer highlighted exemplars

For each category, the exemplars are the gold sentences of that category (minus quoted statute text) plus the coded definition. A sentence's score for a category is its cosine similarity to the nearest exemplar (`PROTOTYPE_MODE = "max"`) or to the category centroid (`"mean"`).

**Dev judgments: leave one judgment out, nested.** For each dev judgment:
1. Inside the remaining dev judgments, run a second leave one out to get honest similarity scores, and pick each category's threshold to maximize F1 on those scores.
2. Build exemplars from all remaining dev judgments, score the held out judgment, apply the thresholds.

**Final model, used on the test judgments and the whole corpus:** exemplars from all dev judgments, thresholds tuned on their leave one out scores.

No sentence is ever scored against an exemplar from its own judgment, and test judgments are never used as exemplars or for tuning.

In [3]:
Y_BY_SID = pd.DataFrame(Y, index=GOLD_IDX, columns=CATS)
QUOTED = set(sents.index[sents.in_quote])

def exemplar_ids(train_docs):
    rows = Y_BY_SID[np.isin(DOCS, train_docs)]
    return {c: [s for s in rows.index[rows[c] == 1] if not (EXCLUDE_QUOTED_EXEMPLARS and s in QUOTED)]
            for c in CATS}

def proto_scores(query_sids, ex):
    Q = X[query_sids]
    S = np.full((len(query_sids), K), -1.0)
    for j, c in enumerate(CATS):
        E = X[ex[c]] if ex[c] else np.zeros((0, X.shape[1]))
        if USE_DEFINITIONS:
            E = np.vstack([E, DEF_X[j:j + 1]])
        if not len(E):
            continue
        if PROTOTYPE_MODE == "mean":
            proto = E.mean(0)
            S[:, j] = Q @ (proto / np.linalg.norm(proto))
        else:
            S[:, j] = (Q @ E.T).max(1)
    return S

def tune_thresholds(S, Yt):
    th = np.full(K, np.inf)
    for j in range(K):
        if Yt[:, j].sum() == 0:
            continue
        best_f = -1
        for t in np.unique(np.quantile(S[:, j], np.linspace(0.3, 0.995, 80))):
            f = f1_score(Yt[:, j], (S[:, j] >= t).astype(int), zero_division=0)
            if f >= best_f:          # ties go to the higher threshold
                best_f, th[j] = f, t
    return th

ROWS_OF = {d: np.where(DOCS == d)[0] for d in gold_docs}
DEV_DOCS = sorted(set(DOCS[~TEST]))

def loo_scores(docs):
    # Score each judgment in docs against exemplars from the other judgments in docs
    return {t: proto_scores(GOLD_IDX[ROWS_OF[t]], exemplar_ids([x for x in docs if x != t])) for t in docs}

def fit(docs):
    S = loo_scores(docs)
    th = tune_thresholds(np.vstack([S[t] for t in docs]), np.vstack([Y[ROWS_OF[t]] for t in docs]))
    return exemplar_ids(docs), th

M2_SCORES = np.zeros((len(Y), K))
M2_PRED = np.zeros((len(Y), K), dtype=int)

for d in DEV_DOCS:
    ex, th = fit([x for x in DEV_DOCS if x != d])
    r = ROWS_OF[d]
    M2_SCORES[r] = proto_scores(GOLD_IDX[r], ex)
    M2_PRED[r] = M2_SCORES[r] >= th

EX_FINAL, TH_FINAL = fit(DEV_DOCS)
r = np.where(TEST)[0]
M2_SCORES[r] = proto_scores(GOLD_IDX[r], EX_FINAL)
M2_PRED[r] = M2_SCORES[r] >= TH_FINAL

print("Dev, leave one judgment out:")
per_category(Y[~TEST], M2_PRED[~TEST]).round(2)


Dev, leave one judgment out:


,precision,recall,f1,support
Ingredients/exceptions,0.10,0.22,0.14,46
Speech vs reputation,0.03,0.60,0.06,5
Procedural,0.08,0.79,0.14,29
Insufficient evidence,0.12,0.40,0.19,25
Abuse of process,0.05,0.08,0.06,24


## Apply M2 to all judgments

This is the final model from above: exemplars from all dev judgments and thresholds tuned on their leave one out scores, so no threshold was picked by looking at the sentences it is applied to.

In [4]:
M2_ALL = (proto_scores(sents.index.values, EX_FINAL) >= TH_FINAL).astype(int)
save_preds("M2 exemplar similarity", M2_PRED, M2_ALL)
print("Share of all sentences per category:", dict(zip(CATS, M2_ALL.mean(0).round(3))))

Share of all sentences per category: {'C1': np.float64(0.195), 'C2': np.float64(0.17), 'C3': np.float64(0.488), 'C4': np.float64(0.079), 'C5': np.float64(0.086)}
